# JW $Z_4^{TF}$ system - cocycle extraction - larger system

Created: 16-09-2026

Objectives:
* Iterate on [this notebook](jw_z_4_tf_system_cocycle_extraction_larger_system.ipynb), for debugging purposes (with ChatGPT), remove the FP even requirement on the disentangling projectors.

# Imports

In [1]:
import numpy as np

In [2]:
import jax
jax.config.update('jax_platform_name', 'cpu')

import jax.numpy as jnp

In [3]:
import matplotlib.pyplot as plt

In [4]:
from tqdm import tqdm

In [5]:
from functools import reduce
from operator import mul
from itertools import product

In [6]:
from random import random

In [7]:
import quimb.tensor as qtn
import quimb as qu

In [8]:
from scipy.stats import unitary_group

In [9]:
from collections import Counter

In [10]:
import pandas as pd

In [11]:
from time import time

In [12]:
from humanize import naturalsize

# Definitions
## Construct cluster state

In [13]:
np_up_X_state = 1/(np.sqrt(2))*np.array([1,1])

In [14]:
qu_up_X_state = qtn.Tensor(
    data=np_up_X_state,
    inds=('k',),
    tags='prod'
)

In [15]:
np_CZ = np.diag([1,1,1,-1])

In [16]:
np_CZ = np_CZ.reshape((2,)*4)

In [17]:
qu_CZ = qtn.Tensor(
    data=np_CZ,
    inds=('k1', 'k2', 'b1', 'b2'),
    tags='CZ'
)

In [18]:
np_hadamard = np.pow(2, -1/2)*np.array([
    [1,1],
    [1,-1]
])

In [19]:
qu_hadamard = qtn.Tensor(data=np_hadamard, inds=('k', 'b'), tags='Had')

In [20]:
def get_cluster_state_qu_tensor_network(num_sites):
    assert (num_sites%2) == 0

    product_state_tensors = [
        qu_up_X_state.reindex({'k': f'kc_1_{i}'})
        for i in range(num_sites)
    ]

    first_layer_circuit_tensors = [
        qu_CZ.reindex({
            'b1': f'kc_1_{i}',
            'b2': f'kc_1_{i+1}',
            'k1': f'kc_2_{i}',
            'k2': f'kc_2_{i+1}'
        })
        for i in range(0, num_sites, 2)
    ]


    second_layer_circuit_tensors = [
        qu_CZ.reindex({
            'b1': f'kc_2_{i}',
            'b2': f'kc_2_{(i+1)%num_sites}',
            'k1': f'kh_{i}',
            'k2': f'k{(i+1)%num_sites}'
        })
        for i in range(1, num_sites+1, 2)
    ]

    hadamard_layer = [
        qu_hadamard.reindex({
            'b': f'kh_{i}',
            'k': f'k{i}'
        })
        for i in range(1, num_sites, 2)
    ]
    all_tensors = (
        product_state_tensors
        + first_layer_circuit_tensors
        + second_layer_circuit_tensors
        + hadamard_layer
    )

    out = qtn.TensorNetwork(all_tensors, virtual=True)
    out.mangle_inner_()

    return out

## Construct product state

In [21]:
np_up_X_state = 1/(np.sqrt(2))*np.array([1,1])

In [22]:
np_up_Z_state = np.array([1,0])

In [23]:
qu_up_Z_state = qtn.Tensor(
    data=np_up_Z_state,
    inds=('k',),
    tags='prod'
)

In [24]:
alternating_states = [
    qu_up_X_state,
    qu_up_Z_state
]

def get_product_qu_tensor_network(num_sites):
    assert (num_sites%2) == 0

    product_state_tensors = [
        alternating_states[i%2].reindex({'k': f'k{i}'})
        for i in range(num_sites)
    ]

    out = qtn.TensorNetwork(
        product_state_tensors,
        virtual=True
    )
    out.mangle_inner_()

    return out

## Symmetries

In [25]:
def multikron(arrays):
    return reduce(np.kron, arrays)

In [26]:
np_I = np.array([
    [1,0],
    [0,1]
])

np_X = np.array([
    [0,1],
    [1,0]
])

np_Y = np.array([
    [0,-1j],
    [1j,0]
])

np_Z = np.array([
    [1,0],
    [0,-1]
])

In [27]:
qu_I = qtn.Tensor(
    np_I,
    inds=['k', 'b'],
    tags='X'
)

qu_X = qtn.Tensor(
    np_X,
    inds=['k', 'b'],
    tags='X'
)

qu_Y = qtn.Tensor(
    np_Y,
    inds=['k', 'b'],
    tags='Y'
)

qu_Z = qtn.Tensor(
    np_Z,
    inds=['k', 'b'],
    tags='Z'
)

In [28]:
def get_multisite_qu_X(num_sites):
    np_many_X = multikron([np_X]*num_sites)

    out = qtn.Tensor(
        np_many_X,
        inds=['k', 'b'],
        tags='mulit_site_X',
    )

    return out

In [29]:
def get_multisite_qu_I(num_sites):
    np_many_I = multikron([np_I]*num_sites)

    out = qtn.Tensor(
        np_many_I,
        inds=['k', 'b'],
        tags='mulit_site_I',
    )

    return out

In [30]:
qu_spin_fermion_fp = (
    qu_I.reindex({'k': 'ks', 'b': 'bs'})
    & qu_Z.reindex({'k': 'kf', 'b': 'bf'})
).contract()

qu_unit_cell_fp = qu_spin_fermion_fp.fuse({
    'k': ['ks', 'kf'],
    'b': ['bs', 'bf']
})

In [31]:
"""
def get_multisite_qu_fp(num_sites):
    assert (num_sites%2)==0

    num_unit_cells = (num_sites//2)

    np
"""

'\ndef get_multisite_qu_fp(num_sites):\n    assert (num_sites%2)==0\n\n    num_unit_cells = (num_sites//2)\n\n    np\n'

Decompose T symmetry as $MK$:

In [32]:
np_00 = np.array([[1,0], [0,0]])
np_11 = np.array([[0,0], [0,1]])

In [33]:
def tensor_product_operators(op_1, op_2):
    out = (
        op_1[..., np.newaxis, np.newaxis]
        *op_2[np.newaxis, np.newaxis, ...]
    )

    return out

In [34]:
np_M = (
    tensor_product_operators(np_X, np_00)
    + tensor_product_operators(np_Y, np_11)
)

In [35]:
qu_M = qtn.Tensor(
    np_M,
    inds=['ks', 'bs', 'kf', 'bf']
)

In [36]:
np_M_reindexed = (
    qu_M
    .fuse({
        'k': ['ks', 'kf'],
        'b': ['bs', 'bf']
    })
    .transpose('k', 'b')
    .data
)

## Extracting projectors

In [37]:
def random_uniform_complex(shape):
    return np.random.uniform(size=shape) + 1j*np.random.uniform(size=shape)

In [38]:
def maximize_projector_states(rho, left_sites, proj_sites, right_sites):
    v = qtn.Tensor(
        data=random_uniform_complex((2,)*(len(proj_sites))),
        inds=[f'k{i}' for i in proj_sites]
    )

    tnopt = qtn.TNOptimizer(
        v,  # the tensor network we want to optimize
        loss_func,  # the function we want to minimize
        norm_fn=normalize_v,
        loss_constants={"rho": rho},
        loss_kwargs={
            "left_sites": left_sites,
            "proj_sites": proj_sites,
            "right_sites": right_sites,
        },
        autodiff_backend="jax",
        optimizer="L-BFGS-B",
        progbar=False
    )

    v_opt = tnopt.optimize(n=2000)

    return v_opt, tnopt.losses

In [39]:
def projector_state_check(state, rho, left_sites):
    left_right_rho = (
        rho
        & state
        & state.conj().reindex({s: f'b{s[1:]}' for s in state.inds})
    )

    left_right_rho = left_right_rho.contract()

    left_inds = [
        f'{s}{i}'
        for i in left_sites
        for s in 'kb'
    ]

    schmidt_decomp = qtn.tensor_core.tensor_split(
        left_right_rho,
        left_inds=left_inds,
        method='svd',
        #cutoff=1e-6,
        cutoff_mode='abs',
        absorb=None,
        renorm=False,
        bond_ind='v'
    )

    schmidt_vals = schmidt_decomp.tensors[1]

    return schmidt_vals

### Embed vector

In [40]:
np_CX = (
    tensor_product_operators(np_00, np_I)
    + tensor_product_operators(np_11, np_X)
)

In [41]:
qu_CX = qtn.Tensor(
    np_CX,
    inds=['k1', 'b1', 'k2', 'b2']
)

In [42]:
np_up_Z_state = np.array([1,0])

In [43]:
qu_up_Z_state = qtn.Tensor(
    data=np_up_Z_state,
    inds=('k',),
    tags='Z0_pad'
)

### Loss function

In [44]:
def get_rho_purity(rho, sites):
    # Assuming rho is a Hermitian reduced density matrix
    reindex_map = (
        {f'k{i}': f'b{i}' for i in sites}
        | {f'b{i}': f'k{i}' for i in sites}
    )

    rho_other = rho.reindex(reindex_map)
    #rho_other.mangle_inner_()
    
    out = (rho & rho_other).contract()

    return out

In [45]:
def normalize_v(v):
    norm = (v & v.conj()).contract()
    w = v*jnp.power(norm, -0.5)
    return w

In [46]:
def loss_func(v, rho, left_sites, proj_sites, right_sites):
    rho_lr = (
        rho
        & v.reindex({f'k{i}': f'b{i}' for i in proj_sites})
        & v.conj()
    )
    rho_lr = rho_lr.contract()
    tr_rho_lr = (
        rho_lr
        .reindex({f'k{i}': f'b{i}' for i in left_sites + right_sites})
        .contract()
    )
    
    rho_l = rho_lr.reindex(
        {f'k{i}': f'b{i}' for i in right_sites}
    )
    rho_l = rho_l.contract()*jnp.power(tr_rho_lr, -0.5)
    
    rho_r = rho_lr.reindex(
        {f'k{i}': f'b{i}' for i in left_sites}
    )
    rho_r = rho_r.contract()*jnp.power(tr_rho_lr, -0.5)
    
    purity_lr = get_rho_purity(rho_lr, left_sites + right_sites)
    purity_l = get_rho_purity(rho_l, left_sites)
    purity_r = get_rho_purity(rho_r, right_sites)
    
    reindex_map = (
        {f'k{i}': f'b{i}' for i in left_sites+right_sites}
        | {f'b{i}': f'k{i}' for i in left_sites+right_sites}
    )
    
    cross_term = (
        rho_lr.reindex(reindex_map)
        & rho_l
        & rho_r
    )
    cross_term = cross_term.contract()

    out = jnp.real(
        (purity_lr + purity_l*purity_r - 2*cross_term)/
        purity_lr
    )

    return out

## Extract cut rho and EDM

In [47]:
def generate_antilinear_edm_from_cut_state(cut_state, sites, num_defect_sites, t_inv=False):
    # Lots of duplicate code, probably a better way to do this.
    # Computes for U(t) or U(t^3)=U(t^-1), the two cases distinguished by t_inv.
    assert 2*num_defect_sites < len(sites)
    assert (num_defect_sites%2) == 0
    assert (len(sites)%2) == 0
    assert (sites[0]%2) == 0

    left_defect_sites = sites[:num_defect_sites]
    right_defect_sites = sites[-num_defect_sites:]
    internal_sites = sites[num_defect_sites:-num_defect_sites]

    if t_inv:
        unitary_part = qu_M.conj()
    else:
        unitary_part = qu_M

    # Being sloppy with the gate indices as the symmetries are invariant
    # under transpose and conjugation.
    left_sym_gates = [
        unitary_part.reindex({
            'ks': f'b{i}',
            'kf': f'b{i+1}',
            'bs': f'c{i}',
            'bf': f'c{i+1}'
        })
        for i in left_defect_sites[::2]
    ]

    inner_gates = [
        unitary_part.reindex({
            'ks': f'k{i}',
            'kf': f'k{i+1}',
            'bs': f'b{i}',
            'bf': f'b{i+1}'
        })
        for i in internal_sites[::2]
    ]

    right_sym_gates = [
        unitary_part.reindex({
            'ks': f'b{i}',
            'kf': f'b{i+1}',
            'bs': f'c{i}',
            'bf': f'c{i+1}'
        })
        for i in right_defect_sites[::2]
    ]

    reindex_map = (
        {
            f'k{i}': f'c{i}'
            for i in (left_defect_sites + right_defect_sites)
        }
        |
        {
            f'k{i}': f'b{i}'
            for i in internal_sites
        }
    )

    # The fact that we don't conjugate the reindexed cut_state means that
    # we are effectively implementing local complex conjugation.
    edm = (
        cut_state
        & cut_state.reindex(reindex_map)
        & left_sym_gates
        & inner_gates
        & right_sym_gates
    )

    edm = edm.contract()

    fuse_maps = [
        ('k_left', (f'k{i}' for i in left_defect_sites)),
        ('b_left', (f'b{i}' for i in left_defect_sites)),
        ('k_right', (f'k{i}' for i in right_defect_sites)),
        ('b_right', (f'b{i}' for i in right_defect_sites))
    ]

    edm.fuse(fuse_maps, inplace=True)

    return edm

In [48]:
def generate_fp_edm_from_cut_state(cut_state, sites, num_defect_sites):
    # Lots of duplicate code, probably a better way to do this.
    # Should really combine with generate_antilinear_edm_from_cut_state
    # Computes for U(t^2)=FP.
    assert 2*num_defect_sites < len(sites)
    assert (num_defect_sites%2) == 0
    assert (len(sites)%2) == 0
    assert (sites[0]%2) == 0

    left_defect_sites = sites[:num_defect_sites]
    right_defect_sites = sites[-num_defect_sites:]
    internal_sites = sites[num_defect_sites:-num_defect_sites]

    unitary_part = qu_spin_fermion_fp

    # Being sloppy with the gate indices as the symmetries are invariant
    # under transpose and conjugation.
    left_sym_gates = [
        unitary_part.reindex({
            'ks': f'b{i}',
            'kf': f'b{i+1}',
            'bs': f'c{i}',
            'bf': f'c{i+1}'
        })
        for i in left_defect_sites[::2]
    ]

    inner_gates = [
        unitary_part.reindex({
            'ks': f'k{i}',
            'kf': f'k{i+1}',
            'bs': f'b{i}',
            'bf': f'b{i+1}'
        })
        for i in internal_sites[::2]
    ]

    right_sym_gates = [
        unitary_part.reindex({
            'ks': f'b{i}',
            'kf': f'b{i+1}',
            'bs': f'c{i}',
            'bf': f'c{i+1}'
        })
        for i in right_defect_sites[::2]
    ]

    reindex_map = (
        {
            f'k{i}': f'c{i}'
            for i in (left_defect_sites + right_defect_sites)
        }
        |
        {
            f'k{i}': f'b{i}'
            for i in internal_sites
        }
    )

    edm = (
        cut_state
        & cut_state.conj().reindex(reindex_map)
        & left_sym_gates
        & inner_gates
        & right_sym_gates
    )

    edm = edm.contract()

    fuse_maps = [
        ('k_left', (f'k{i}' for i in left_defect_sites)),
        ('b_left', (f'b{i}' for i in left_defect_sites)),
        ('k_right', (f'k{i}' for i in right_defect_sites)),
        ('b_right', (f'b{i}' for i in right_defect_sites))
    ]

    edm.fuse(fuse_maps, inplace=True)

    return edm

In [49]:
def generate_all_edms_from_cut_state(cut_state, sites, num_defect_sites):
    args = (cut_state, sites, num_defect_sites)

    out = [
        generate_antilinear_edm_from_cut_state(*args, False),
        generate_fp_edm_from_cut_state(*args),
        generate_antilinear_edm_from_cut_state(*args, True)
    ]

    return out

## Defect operators

In [50]:
def random_uniform_complex(shape):
    return np.random.uniform(size=shape) + 1j*np.random.uniform(size=shape)

In [51]:
def solve_for_boundary_operators(edm, num_iters=100):
    # Careful, the indices are reversed here for ease.
    # i.e. the k, b indices have been swapped to make tensor contraction easier.
    scores = list()

    u_left = qtn.tensor_builder.rand_tensor(
        (edm.ind_size('b_left'), edm.ind_size('k_left')),
        inds=['k_left', 'b_left'],
        dtype='complex64'
    )

    u_right = qtn.tensor_builder.rand_tensor(
        (edm.ind_size('b_right'), edm.ind_size('k_right')),
        inds=['k_right', 'b_right'],
        dtype='complex64'
    )

    for _ in range(num_iters):
        right_edm = (edm & u_left).contract()
        data = right_edm.data
        U, S, VH = np.linalg.svd(data)
        scores.append(np.sum(S))
    
        sol = (U @ VH).conj().T
        u_right = qtn.Tensor(sol, inds = ['b_right', 'k_right'])

        left_edm = (edm & u_right).contract()
        data = left_edm.data
        U, S, VH = np.linalg.svd(data)
        scores.append(np.sum(S))
    
        sol = (U @ VH).conj().T
        u_left = qtn.Tensor(sol, inds = ['b_left', 'k_left'])

    return (u_left, u_right), scores

## Apply random unitary to groundstate

In [52]:
def generate_random_su2():
    # Randomly sample a unitary, and scale by the determinant.
    u = unitary_group.rvs(2)
    det_u = np.linalg.det(u)
    su = u*np.power(det_u, -0.5)

    return su

In [53]:
X =  generate_random_su2()

In [54]:
np.linalg.det(X)

np.complex128(1+5.551115123125782e-17j)

In [55]:
np.round(X @ (X.conj().T), 3)

array([[1.+0.j, 0.+0.j],
       [0.-0.j, 1.+0.j]])

In [56]:
def generate_random_symmetry_respecting_unitary_no_offset():
    # Generate a unitary which commutes with MK, where
    # M = X tensor (|0><0|) + Y tensor (|1><1|)

    phi = np.random.uniform(0, 2*np.pi)
    phi_phasor = np.exp(1j*phi)
    u0 = np.diag([phi_phasor, phi_phasor.conj()])

    if random() > 0.5:
        u0 = u0 @ np_X

    u1 = generate_random_su2()

    u = (
        tensor_product_operators(u0, np_00)
        + tensor_product_operators(u1, np_11)
    )

    qu_u = qtn.Tensor(
        u,
        inds=['ks', 'bs', 'kf', 'bf']
    )
    
    return qu_u

In [57]:
def generate_random_symmetry_respecting_unitary_offset():
    # Generate a unitary U such that IUI commutes with (MM)K, where
    # M = X tensor (|0><0|) + Y tensor (|1><1|), and concatenation denotes
    # tensor product.
    phi = np.random.uniform(0, 2*np.pi)
    phi_phasor = np.exp(1j*phi)
    u0 = np.diag([phi_phasor, phi_phasor.conj()])

    phi = np.random.uniform(0, 2*np.pi)
    phi_phasor = np.exp(1j*phi)
    u1 = np.diag([phi_phasor, phi_phasor.conj()])

    u = (
        tensor_product_operators(np_00, u0)
        + tensor_product_operators(np_11, u1)
    )

    qu_u = qtn.Tensor(
        u,
        inds=['kf', 'bf', 'ks', 'bs']
    )
    
    return qu_u

In [58]:
def generate_random_symmetry_respecting_unitary(offset):
    if offset:
        return generate_random_symmetry_respecting_unitary_offset()
    else:
        return generate_random_symmetry_respecting_unitary_no_offset()

In [59]:
# Warning, likely making assupmtions about shape of psi, number of sites being even here etc.
def apply_haar_random_fdlu_to_quimb_state(psi, domains_dict):
    num_sites = domains_dict['num_system_sites']

    depth = domains_dict['fdlu_depth']
    offset = domains_dict['fdlu_offset']
    all_circuit_lists = [
        list() for _ in range(depth)
    ]

    for layer, circuit_list in enumerate(all_circuit_lists):
        delta = layer
        is_offset = ((offset + delta)%2 == 1)

        for i in range(num_sites//2):
            site_1 = ((2*i)+delta+offset)%num_sites
            site_2 = ((2*i)+1+delta+offset)%num_sites

            u = generate_random_symmetry_respecting_unitary(is_offset)

            if is_offset:
                reindex_map = {
                    'kf': f'k_{layer+1}_{site_1}',
                    'ks': f'k_{layer+1}_{site_2}',
                    'bf': f'k_{layer}_{site_1}',
                    'bs': f'k_{layer}_{site_2}'
                }
            else:
                reindex_map = {
                    'ks': f'k_{layer+1}_{site_1}',
                    'kf': f'k_{layer+1}_{site_2}',
                    'bs': f'k_{layer}_{site_1}',
                    'bf': f'k_{layer}_{site_2}'
                }
            
            qu_u = u.reindex(reindex_map)

            circuit_list.append(qu_u)

    all_tensors = (
        [psi.reindex({f'k{i}': f'k_0_{i}' for i in range(num_sites)})]
        + sum(all_circuit_lists, start=[])
    )

    out = (
        qtn
        .TensorNetwork(all_tensors, virtual=False)
        .mangle_inner_()
        .reindex({f'k_{depth}_{i}': f'k{i}' for i in range(num_sites)}) 
    )

    return out

In [60]:
def extract_time_reversal_information_after_random_fdlu(psi, domains_dict,
    num_random_states=20):

    out = list()

    for _ in range(num_random_states):
        rand_psi = apply_haar_random_fdlu_to_quimb_state(psi, domains_dict)
        out.append(extract_time_reversal_information(rand_psi, domains_dict))

    return out

In [61]:
def extract_factorization_time_reversal_information_after_random_fdlu(psi,
    domains_dict, num_random_states=20):
    out = list()

    for _ in range(num_random_states):
        rand_psi = apply_haar_random_fdlu_to_quimb_state(psi, domains_dict)
        data = extract_factorization_time_reversal_information(
            rand_psi,
            domains_dict
        )
        out.append(data)

    return out

In [62]:
def get_quimb_psi_from_quspin_psi(quspin_psi):
    quimb_psi = qtn.Tensor(
        quspin_psi[::-1].reshape((2,)*num_sites),
        inds=[f'k{i}' for i in range(num_sites)]
    )

    return quimb_psi

## Sweep function

In [63]:
def extract_projector(psi, rho_sites, num_pad_sites, jw_even=False):
    proj_sites = rho_sites
    left_sites = list(range(
        min(rho_sites) - num_pad_sites,
        min(rho_sites)
    ))
    right_sites = list(range(
        max(rho_sites)+1,
        max(rho_sites)+num_pad_sites+1
    ))
    all_sites = left_sites + proj_sites + right_sites
    rho = (psi & psi.conj().reindex({f'k{i}': f'b{i}' for i in all_sites}))
    proj_vec, losses = maximize_projector_states(
        rho,
        left_sites,
        proj_sites,
        right_sites
    )

    schmidt_vals = projector_state_check(
        proj_vec,
        rho,
        left_sites
    )

    return proj_vec, losses, schmidt_vals

In [64]:
def find_invariants_via_projectors_from_random_state(psi, domains_dict, jw_even=False):
    rand_psi = apply_haar_random_fdlu_to_quimb_state(psi, domains_dict)
    
    left_proj_vec, *left_proj_vec_results = extract_projector(
        rand_psi,
        domains_dict['left_projector_sites'],
        domains_dict['num_projector_pad_sites'],
        jw_even
    )
    
    right_proj_vec, *right_proj_vec_results = extract_projector(
        rand_psi,
        domains_dict['right_projector_sites'],
        domains_dict['num_projector_pad_sites'],
        jw_even
    )
    
    cut_sites = list(range(
        min(domains_dict['left_projector_sites']),
        max(domains_dict['right_projector_sites'])+1
    ))
    
    cut_rho_unprojected = (
        rand_psi
        & rand_psi.conj().reindex({f'k{i}': f'b{i}' for i in cut_sites})
    )
    
    left_proj_sites = domains_dict['left_projector_sites']
    right_proj_sites = domains_dict['right_projector_sites']
    
    cut_rho = (
        cut_rho_unprojected.reindex({
            **{f'k{i}': f'l{i}' for i in left_proj_sites},
            **{f'k{i}': f'l{i}' for i in right_proj_sites},
            **{f'b{i}': f'c{i}' for i in left_proj_sites},
            **{f'b{i}': f'c{i}' for i in right_proj_sites}
        })
        & left_proj_vec.conj().reindex({f'k{i}': f'l{i}' for i in left_proj_sites})
        & left_proj_vec
        & left_proj_vec.reindex({f'k{i}': f'c{i}' for i in left_proj_sites})
        & left_proj_vec.conj().reindex({f'k{i}': f'b{i}' for i in left_proj_sites})
        & right_proj_vec.conj().reindex({f'k{i}': f'l{i}' for i in right_proj_sites})
        & right_proj_vec
        & right_proj_vec.reindex({f'k{i}': f'c{i}' for i in right_proj_sites})
        & right_proj_vec.conj().reindex({f'k{i}': f'b{i}' for i in right_proj_sites})
    )
    
    cut_rho_trace = (
        cut_rho
        .reindex({f'b{i}': f'k{i}' for i in cut_sites})
        .contract()
    )
    
    cut_rho = cut_rho/cut_rho_trace
    
    tranpose_map = (
        {f'k{i}': f'b{i}' for i in cut_sites}
        | {f'b{i}': f'k{i}' for i in cut_sites}
    )
    
    cut_rho_purity = (
        (cut_rho & cut_rho.reindex(tranpose_map))
        .contract()
    )
    
    sub_cut_sites = list(range(
        max(domains_dict['left_projector_sites'])+1,
        min(domains_dict['right_projector_sites'])
    ))
    
    sub_cut_rho = (
        cut_rho
        .reindex({f'k{i}': f'b{i}' for i in left_proj_sites + right_proj_sites})
        #.contract()
    )
    
    sub_cut_rho_trace = (
        sub_cut_rho
        .reindex({f'b{i}': f'k{i}' for i in sub_cut_sites})
        .contract()
    )
    
    tranpose_map = (
        {f'k{i}': f'b{i}' for i in sub_cut_sites}
        | {f'b{i}': f'k{i}' for i in sub_cut_sites}
    )
    
    sub_cut_rho_purity = (
        (sub_cut_rho & sub_cut_rho.reindex(tranpose_map))
        .contract()
    )
    
    cut_score, cut_state, cut_overlap = get_dominant_eigenvector(sub_cut_rho, False)

    cut_state_fermion_parity = compute_fermion_parity(cut_state, sub_cut_sites)

    edms = generate_all_edms_from_cut_state(
        cut_state,
        sub_cut_sites,
        domains_dict['num_defect_sites']
    )

    defect_ops_results = [
        solve_for_boundary_operators(
            edm,
            num_iters=20
        )
        for edm in edms
    ]

    left_defect_sites = list(range(
        max(domains_dict['left_projector_sites']) + 1,
        max(domains_dict['left_projector_sites']) + 1 + domains_dict['num_defect_sites']
    ))
    right_defect_sites = list(range(
        min(domains_dict['right_projector_sites']) - domains_dict['num_defect_sites'],
        min(domains_dict['right_projector_sites'])
    ))
    
    left_rdm = (
        cut_rho
        .reindex({
            f'b{i}': f'k{i}'
            for i in cut_sites if i not in left_defect_sites
        })
    )
    left_rdm = left_rdm.contract()
    left_fuse_map = [
        ('k_left', [f'k{i}' for i in left_defect_sites]),
        ('b_left', [f'b{i}' for i in left_defect_sites])
    ]
    left_rdm.fuse(left_fuse_map, inplace=True)
    
    right_rdm = (
        cut_rho
        .reindex({
            f'b{i}': f'k{i}'
            for i in cut_sites if i not in right_defect_sites
        })
    )
    right_rdm = right_rdm.contract()
    right_fuse_map = [
        ('k_right', [f'k{i}' for i in right_defect_sites]),
        ('b_right', [f'b{i}' for i in right_defect_sites])
    ]
    right_rdm.fuse(right_fuse_map, inplace=True)

    defect_ops_pairs = [l[0] for l in defect_ops_results]
    left_defect_ops, right_defect_ops = list(zip(*defect_ops_pairs))

    np_left_rdm = (
        left_rdm
        .transpose('k_left', 'b_left')
        .data
    )

    np_left_defect_ops = [
        left_defect_op.transpose('k_left', 'b_left').data.T
        for left_defect_op in left_defect_ops
    ]

    np_right_rdm = (
        right_rdm
        .transpose('k_right', 'b_right')
        .data
    )
    
    np_right_defect_ops = [
        right_defect_op.transpose('k_right', 'b_right').data.T
        for right_defect_op in right_defect_ops
    ]

    np_left_defect_ops = [
        np.identity(np_left_defect_ops[0].shape[0]),
        *np_left_defect_ops
    ]
    np_right_defect_ops = [
        np.identity(np_right_defect_ops[0].shape[0]),
        *np_right_defect_ops
    ]

    """
    #Ignore for now
    if domains_dict['normalize_defect_op']:
        np_left_defect_op = normalize_defect_op(
            np_left_defect_op,
            np_left_rdm
        )
        right_defect_op = normalize_defect_op(
            np_right_defect_op,
            np_right_rdm
        )
    """

    fp_list = [np_I, np_Z]
    np_fp = multikron([
        fp_list[i%2]
        for i in range(domains_dict['num_defect_sites'])
    ])

    left_defect_op_invariant = np.trace(
        np_fp
        @ np_left_defect_ops[1].conj().T
        @ np_fp
        @ np_left_defect_ops[1]
        @ np_left_rdm
    )

    right_defect_op_invariant = np.trace(
        np_fp
        @ np_right_defect_ops[1].conj().T
        @ np_fp
        @ np_right_defect_ops[1]
        @ np_right_rdm
    )

    np_sym_defect_m = get_multisite_op(
        np_M_reindexed,
        domains_dict['num_defect_sites']
    )

    np_defect_I = multikron([
        np_I
        for i in range(domains_dict['num_defect_sites'])
    ])

    unitary_sym_op_list = [
        np_defect_I,
        np_sym_defect_m,
        np_fp,
        np_sym_defect_m.conj()
    ]

    left_cocyles = get_cocycles_from_rho(
        np_left_rdm,
        np_left_defect_ops,
        unitary_sym_op_list
    )
    left_cocyles = gauge_fix_cocycles(left_cocyles)

    right_cocyles = get_cocycles_from_rho(
        np_right_rdm,
        np_right_defect_ops,
        unitary_sym_op_list
    )
    right_cocyles = gauge_fix_cocycles(right_cocyles)

    all_cocyles = np.array([left_cocyles, right_cocyles])

    cocycle_equation_output = compute_cocyle_equation_from_cocycles(all_cocyles)
    bosonic_cocycle_equation_output = (
        compute_bosonic_cocyle_equation_from_cocycles(all_cocyles)
    )

    cocycle_invariants = (all_cocyles[:,0,0]**2)*(all_cocyles[:,1,1])

    fermion_decoration_from_cocycle = all_cocyles[:, 0, 1]/all_cocyles[:, 1, 0]
    """
    left_bosonic_cocycle = get_bosonic_cocycle(
        np_left_rdm,
        np_left_defect_ops[1],
        unitary_sym_op_list
    )
    right_bosonic_cocycle = get_bosonic_cocycle(
        np_right_rdm,
        np_right_defect_ops[1],
        unitary_sym_op_list
    )

    left_bosonic_cocyle_eq_from_boosnic_cocycle = (
        compute_bosonic_cocyle_equation_from_bosonic_cocycle(
            left_bosonic_cocycle
        )
    )
    right_bosonic_cocyle_eq_from_boosnic_cocycle = (
        compute_bosonic_cocyle_equation_from_bosonic_cocycle(
            right_bosonic_cocycle
        )
    )
    """

    out = {
        'left_proj_vec': left_proj_vec,
        'left_proj_vec_results': left_proj_vec_results,
        'right_proj_vec': right_proj_vec,
        'right_proj_vec_results': right_proj_vec_results,
        'cut_rho_trace': cut_rho_trace,
        'cut_rho_purity': cut_rho_purity,
        'sub_cut_rho_trace': sub_cut_rho_trace,
        'sub_cut_rho_purity': sub_cut_rho_purity,
        'cut_score': cut_score,
        'cut_state': cut_state,
        'cut_state_fermion_parity': cut_state_fermion_parity,
        'cut_overlap': cut_overlap,
        'defect_ops_scores': [l[1] for l in defect_ops_results],
        'left_defect_ops': left_defect_ops,
        'right_defect_ops': right_defect_ops,
        'left_defect_op_invariant': left_defect_op_invariant,
        'right_defect_op_invariant': right_defect_op_invariant,
        'cocycles': all_cocyles,
        'cocycle_equation_output': cocycle_equation_output,
        'bosonic_cocycle_equation_output': bosonic_cocycle_equation_output,
        'cocycle_invariants': cocycle_invariants,
        'fermion_decoration_from_cocycle': fermion_decoration_from_cocycle
    }

    return out

## Find dominant eigenvector

In [65]:
def random_uniform_complex(shape):
    return np.random.uniform(size=shape) + 1j*np.random.uniform(size=shape)

In [66]:
def lanczos_iteration(rho, v, sites):
    w = (
        rho & v.reindex({f'k{i}': f'b{i}' for i in sites})
    ).contract()

    w_norm = np.sqrt((w & w.conj()).contract())

    out = w/w_norm

    return out

In [67]:
def multiply_state_by_jw_string(state, sites):
    gates = [
        qu_Z.reindex({'k': f'k{i}', 'b': f'b{i}'})
        for i in sites if (i%2 == 1)
    ]

    out = qtn.TensorNetwork(
        [
            *gates,
            state.reindex({f'k{i}': f'b{i}' for i in sites if (i%2 == 1)})
        ]
    )

    return out.contract()

In [68]:
def lanczos_iteration_jw_even(rho, v, sites):
    w = multiply_state_by_jw_string(v, sites)
    w = (v+w)/2

    w = (
        rho & w.reindex({f'k{i}': f'b{i}' for i in sites})
    ).contract()

    w = (multiply_state_by_jw_string(w, sites) + w)/2

    w_norm = np.sqrt((w & w.conj()).contract())

    out = w/w_norm

    return out

In [69]:
def lanczos_algorithm(rho, v, sites, num_iters=20, jw_even=False):
    update_func = lanczos_iteration_jw_even if jw_even else lanczos_iteration
    for _ in range(num_iters):
        v = update_func(rho, v, sites)

    return v

In [70]:
def get_dominant_eigenvector(rho, jw_even=False):
    k_inds = [i for i in rho.outer_inds() if i.startswith('k')]
    #b_inds = [i for i in rho.inds if i.startswith('b')]

    sites = [int(s[1:]) for s in k_inds]

    v = qtn.Tensor(
        data=random_uniform_complex((2,)*len(sites)),
        inds=[f'k{i}' for i in sites]
    )

    v = lanczos_algorithm(
        rho,
        v,
        sites,
        num_iters=20,
        jw_even=jw_even
    )

    update_func = lanczos_iteration_jw_even if jw_even else lanczos_iteration
    new_v = update_func(rho, v, sites)

    overlap = np.abs((v & new_v.conj()).contract())

    score = (
        v.reindex({f'k{i}': f'b{i}' for i in sites})
        & rho
        & v.conj()
    )
    score = score.contract()


    return score, v, overlap

In [71]:
def compute_fermion_parity(state, sites):
    odd_sites = [i for i in sites if (i%2 == 1)]

    gates = [
        qu_Z.reindex({'k': f'k{i}', 'b': f'b{i}'})
        for i in odd_sites
    ]

    gate_exp = (
        state.reindex({f'k{i}': f'b{i}' for i in odd_sites})
        & gates
        & state.conj()
    ).contract()

    return gate_exp

## Extract cocyles

In [72]:
def get_multisite_op(op, num_defect_sites):
    assert (num_defect_sites%2 == 0)

    np_op = multikron([op,]*(num_defect_sites//2))

    out = qtn.Tensor(
        np_op,
        inds=['k', 'b']
    )

    out = out.transpose('k', 'b').data

    return out

In [73]:
def normalize_defect_op(defect_op, defect_rho):
    exp = np.trace(
        defect_op
        @ defect_op
        @ defect_rho
    )

    phase = np.power(exp/np.abs(exp), -0.5)

    out = defect_op*phase

    return out

In [74]:
def get_t_squared_defect_op_from_t_defect_op(defect_op, unitary_sym_op):
    out = (
        unitary_sym_op
        @ (defect_op.conj())
        @ (unitary_sym_op.conj().T)
        @ defect_op
    )

    return out

In [75]:
def get_t_inverse_defect_op_from_t_defect_op(defect_op, unitary_sym_op):
    out = (
        (unitary_sym_op.T)
        @ (defect_op.T)
        @ (unitary_sym_op.conj())
    )

    return out

In [76]:
def get_all_defect_ops_from_t_defect_op(defect_op, unitary_sym_op):
    out = [
        np.identity(defect_op.shape[0]),
        defect_op,
        get_t_squared_defect_op_from_t_defect_op(defect_op, unitary_sym_op),
        get_t_inverse_defect_op_from_t_defect_op(defect_op, unitary_sym_op)
    ]

    return out

In [77]:
def get_cocycle(defect_rho, defect_op_list, unitary_sym_op_list, g_index,
                  h_index):
    g_op = defect_op_list[g_index]

    h_op = defect_op_list[h_index]
    h_op = h_op.conj() if (g_index%2 == 1) else h_op

    gh_index = (g_index+h_index)%4
    gh_op = defect_op_list[gh_index]

    unitary_sym_op = unitary_sym_op_list[g_index]

    out = np.trace(
        (gh_op.conj().T)
        @ unitary_sym_op
        @ h_op
        @ (unitary_sym_op.conj().T)
        @ g_op
        @ defect_rho
    )

    return out

In [78]:
def get_cocycles_from_rho(defect_rho, defect_op_list, unitary_sym_op_list):
    out = np.array([
        [
            get_cocycle(defect_rho, defect_op_list, unitary_sym_op_list, i, j)
            for j in range(1,4)
        ]
        for i in range(1,4)
    ])

    return out

In [79]:
def gauge_fix_cocycles(cocycles):
    # Gauge fix cocycles to match appendix A.2 of Turzillo & You 2019
    L = np.array([
        1,
        1,
        np.power(cocycles[1,1], 0.5),
        np.power(cocycles[1,1], 0.5)/cocycles[1,0]
    ])

    gauge_transformation = np.ones((3,3), dtype=complex)

    for g, h in product(range(1,4), repeat=2):
        num = L[(g+h)%4]
        if (g%2 == 1):
            denom = (L[h].conj())*L[g]
        else:
            denom = L[h]*L[g]
        gauge_transformation[(g-1), (h-1)] = num/denom

    out = cocycles*gauge_transformation

    return out

In [80]:
def compute_cocyle_equation_from_cocycles(cocyles):
    shape = cocyles
    # A bit janky, oh well
    X = np.ones((2, 4, 4), dtype=complex)

    X[:, 1:, 1:] = cocyles

    out = np.zeros((2, 4, 4, 4), dtype=complex)

    for i,j,k in product(range(4), repeat=3):
        num = X[:,i,j]*X[:,(i+j)%4, k]
        if (i%2 == 0):
            denom = X[:,j,k]*X[:, i, (j+k)%4]
        if (i%2 == 1):
            denom = (X[:,j,k].conj())*X[:,i,(j+k)%4]

        #print(num)
        #print(denom)

        out[:,i,j,k] = num/denom

    return out

In [81]:
def compute_bosonic_cocyle_equation_from_cocycles(cocyles):
    bosonic_cocycles = cocyles[..., :1, :1]
    shape = bosonic_cocycles
    # A bit janky, oh well
    X = np.ones((2, 2, 2), dtype=complex)

    X[:, 1:, 1:] = bosonic_cocycles

    out = np.zeros((2, 2, 2, 2), dtype=complex)

    for i,j,k in product(range(2), repeat=3):
        num = X[:,i,j]*X[:,(i+j)%2, k]
        if (i == 0):
            denom = X[:,j,k]*X[:, i, (j+k)%2]
        if (i == 1):
            denom = (X[:,j,k].conj())*X[:,i,(j+k)%2]

        out[:,i,j,k] = num/denom

    return out

In [82]:
def get_bosonic_cocycle(defect_rho, defect_op, unitary_sym_op_list):
    defect_op_list = get_all_defect_ops_from_t_defect_op(
        defect_op,
        unitary_sym_op_list[1]
    )

    defect_op = defect_op_list[1]
    fp_defect_op = defect_op_list[2]

    unitary_sym_op = unitary_sym_op_list[1]
    fp_sym_op = unitary_sym_op_list[2]

    out = np.trace(
        (fp_defect_op.conj().T)
        @ unitary_sym_op
        @ defect_op.conj()
        @ (unitary_sym_op.conj().T)
        @ defect_op
        @ defect_rho
    )

    return out

In [83]:
def compute_bosonic_cocyle_equation_from_bosonic_cocycle(bosonic_cocycle):
    X = np.ones((2, 2), dtype=complex)

    X[1, 1] = bosonic_cocycle

    out = np.zeros((2, 2, 2), dtype=complex)

    for i,j,k in product(range(2), repeat=3):
        num = X[i,j]*X[(i+j)%2, k]
        if (i == 0):
            denom = X[j,k]*X[i, (j+k)%2]
        if (i == 1):
            denom = (X[j,k].conj())*X[i,(j+k)%2]

        out[i,j,k] = num/denom

    return out

# Check random unitaries and symmetry

In [84]:
domains_dict = {
    'num_system_sites': 24,
    'left_projector_sites': list(range(4, 8)),
    'right_projector_sites': list(range(16, 20)),
    'num_projector_pad_sites': 2,
    'num_defect_sites': 2,
    'fdlu_depth': 2,
    'fdlu_offset': 1
}

In [85]:
cluster_psi = get_cluster_state_qu_tensor_network(domains_dict['num_system_sites'])

In [86]:
rand_psi = apply_haar_random_fdlu_to_quimb_state(cluster_psi, domains_dict)

In [87]:
(rand_psi & rand_psi.conj()).contract()

np.complex128(0.9999999999999971-2.42861286636753e-17j)

In [88]:
(rand_psi & cluster_psi.conj()).contract()

np.complex128(0.00026794556347100755-1.3552527156068805e-20j)

In [89]:
symmetry_gates = [
    qu_M.reindex({
        'ks': f'k{i}', 'bs':f'b{i}',
        'kf': f'k{i+1}', 'bf':f'b{i+1}',
    })
    for i in range(0, domains_dict['num_system_sites'], 2)
]

In [90]:
sym_cluster_psi = qtn.TensorNetwork(
    [
        rand_psi.reindex({f'k{i}': f'b{i}' for i in range(domains_dict['num_system_sites'])}),
        *symmetry_gates
    ]
)

In [91]:
(
    sym_cluster_psi & sym_cluster_psi.conj()
).contract()

np.complex128(0.999999999999997+4.163336342344337e-17j)

In [92]:
(
    sym_cluster_psi & rand_psi.conj()
).contract()

np.complex128(-0.018291712114264377+9.324138683375338e-18j)

In [93]:
(
    sym_cluster_psi & rand_psi
).contract()

np.complex128(0.999999999999997+5.204170427930421e-17j)

In [94]:
product_psi = get_product_qu_tensor_network(domains_dict['num_system_sites'])

In [95]:
rand_psi = apply_haar_random_fdlu_to_quimb_state(product_psi, domains_dict)

In [96]:
(rand_psi & rand_psi.conj()).contract()

np.complex128(0.9999999999999976+2.7755575615628914e-17j)

In [97]:
(rand_psi & product_psi.conj()).contract()

np.complex128(0.0036780668449670705+1.4708365954196363e-18j)

In [98]:
symmetry_gates = [
    qu_M.reindex({
        'ks': f'k{i}', 'bs':f'b{i}',
        'kf': f'k{i+1}', 'bf':f'b{i+1}',
    })
    for i in range(0, domains_dict['num_system_sites'], 2)
]

In [99]:
sym_product_psi = qtn.TensorNetwork(
    [
        rand_psi.reindex({f'k{i}': f'b{i}' for i in range(domains_dict['num_system_sites'])}),
        *symmetry_gates
    ]
)

In [100]:
(
    sym_product_psi & sym_product_psi.conj()
).contract()

np.complex128(0.9999999999999976+2.7755575615628914e-17j)

In [101]:
(
    sym_product_psi & rand_psi.conj()
).contract()

np.complex128(-0.00014605613183265686+4.0657581468206416e-19j)

In [102]:
(
    sym_product_psi & rand_psi
).contract()

np.complex128(0.9999999999999976+2.7755575615628914e-17j)

So random unitaries are symmetric, nice.

In [103]:
import cProfile
import pstats
from time import perf_counter

prof = cProfile.Profile()
start = perf_counter()

prof.enable()
try:
    current = find_invariants_via_projectors_from_random_state(
        cluster_psi, domains_dict, jw_even=True
    )
finally:
    prof.disable()
    print(f"Elapsed: {perf_counter() - start:.2f} s")
    pstats.Stats(prof).strip_dirs().sort_stats("cumulative").print_stats(30)

print("Projector loss evaluations:",
      len(current["left_proj_vec_results"][0]),
      len(current["right_proj_vec_results"][0]))

Elapsed: 2.45 s
         2369039 function calls (2309859 primitive calls) in 2.438 seconds

   Ordered by: cumulative time
   List reduced from 3129 to 30 due to restriction <30>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
    84/21    0.000    0.000    1.561    0.074 traceback_util.py:191(reraise_with_filtered_traceback)
    60/11    0.000    0.000    1.332    0.121 pjit.py:243(cache_miss)
    60/57    0.000    0.000    1.235    0.022 pjit.py:128(_run_python_pjit)
   391/44    0.108    0.000    1.156    0.026 profiler.py:424(wrapper)
        2    0.000    0.000    1.107    0.554 4113890469.py:1(maximize_projector_states)
       11    0.000    0.000    1.091    0.099 pjit.py:1124(_pjit_call_impl_python)
        2    0.000    0.000    1.088    0.544 optimize.py:1488(optimize)
        2    0.000    0.000    1.088    0.544 optimize.py:1526(optimize_scipy)
        2    0.000    0.000    0.989    0.494 _minimize.py:54(minimize)
        2    0.000    0.000    0.9

# Cluster state - depth 2

In [104]:
domains_dict = {
    'num_system_sites': 32,
    'left_projector_sites': list(range(4, 8)),
    'right_projector_sites': list(range(22, 28)),
    'num_projector_pad_sites': 2,
    'num_defect_sites': 4,
    'fdlu_depth': 2,
    'fdlu_offset': 0
}

In [105]:
cluster_psi = get_cluster_state_qu_tensor_network(domains_dict['num_system_sites'])

In [106]:
results = list()

for _ in tqdm(range(20)):
    current = find_invariants_via_projectors_from_random_state(
        cluster_psi,
        domains_dict,
        jw_even=True
    )
    results.append(current)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:27<00:00,  1.39s/it]


### Analyze results

#### Projector scores

In [107]:
np.round(np.array([
    d['left_proj_vec_results'][0][-1]
    for d in results
]), 3)

array([-0.,  0.,  0.,  0.,  0., -0.,  0.,  0., -0., -0., -0., -0.,  0.,
        0.,  0.,  0., -0.,  0.,  0.,  0.])

In [108]:
np.round(np.array([
    d['right_proj_vec_results'][0][-1]
    for d in results
]), 3)

array([ 0., -0.,  0.,  0.,  0.,  0.,  0., -0.,  0.,  0., -0.,  0.,  0.,
       -0.,  0.,  0., -0.,  0.,  0.,  0.])

In [109]:
def get_schmidt_vals_ratio(schmidt_vals):
    if len(schmidt_vals.data) > 1:
        return schmidt_vals.data[1]/schmidt_vals.data[0]
    else:
        return 0

In [110]:
np.round(np.array([
    get_schmidt_vals_ratio(d['left_proj_vec_results'][1])
    for d in results
]), 3)

array([0.114, 0.238, 0.089, 0.118, 0.223, 0.017, 0.696, 0.349, 0.28 ,
       0.22 , 0.294, 0.277, 0.321, 0.531, 0.329, 0.129, 0.442, 0.264,
       0.173, 0.215])

In [111]:
np.round(np.array([
    get_schmidt_vals_ratio(d['right_proj_vec_results'][1])
    for d in results
]), 3)

array([0.134, 0.163, 0.416, 0.332, 0.388, 0.266, 0.165, 0.282, 0.251,
       0.608, 0.371, 0.083, 0.487, 0.473, 0.157, 0.207, 0.449, 0.576,
       0.078, 0.322])

#### Purities

In [112]:
np.round(np.array(
    [d['cut_rho_purity'] for d in results]
), 3)

array([1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j,
       1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.-0.j,
       1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j])

In [113]:
np.round(np.array(
    [d['sub_cut_rho_trace'] for d in results]
), 3)

array([1.+0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j,
       1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j,
       1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j])

In [114]:
np.round(np.array(
    [d['sub_cut_rho_purity'] for d in results]
), 3)

array([1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j,
       1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.-0.j,
       1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j])

Purities are the same, which one could likely prove.

In [115]:
np.round(np.array(
    [d['sub_cut_rho_purity'] for d in results]
), 3)

array([1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j,
       1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.-0.j,
       1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j])

#### Cut state results

In [116]:
np.array(
    [d['cut_state_fermion_parity'] for d in results]
)

array([ 0.42846942-3.61330247e-19j, -0.0466365 -1.85605260e-19j,
       -0.7989019 +4.18637249e-20j,  0.37528692-1.66835909e-18j,
       -0.02422407+7.06261329e-19j, -0.70747182+4.65575455e-19j,
        0.04512103-1.51126747e-19j,  0.08574967+7.82970064e-19j,
        0.19543369-3.68995642e-19j,  0.17427493-6.41627577e-19j,
        0.36409251+2.70112785e-19j,  0.05101669-1.51898125e-19j,
       -0.25015853-1.56179627e-19j, -0.37801615+2.14787547e-19j,
        0.15987857+3.47186207e-20j,  0.18154574-1.20303383e-18j,
        0.00216981+6.21387839e-19j,  0.06255068-1.42508803e-18j,
       -0.33335185+1.76387642e-19j, -0.08644061-4.33534611e-19j])

So the FP of the cut state can be even or odd. Interesting!

In [117]:
np.round(np.array(
    [d['cut_overlap'] for d in results]
), 3)

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1.])

#### Defect op scores overlaps

In [118]:
np.round(np.array([
    [d['defect_ops_scores'][i][-1] for d in results]
    for i in range(3)
]), 5)

array([[0.99763, 0.49523, 0.99983, 0.91836, 0.99997, 0.99872, 0.99655,
        0.9999 , 0.9895 , 0.99829, 0.98681, 1.     , 1.     , 0.9945 ,
        0.9997 , 0.98044, 0.99697, 0.98279, 0.9997 , 0.99967],
       [1.     , 1.     , 1.     , 1.     , 1.     , 1.     , 1.     ,
        1.     , 1.     , 1.     , 1.     , 1.     , 1.     , 1.     ,
        1.     , 1.     , 1.     , 1.     , 1.     , 1.     ],
       [0.99763, 0.49523, 0.99983, 0.91836, 0.99997, 0.99872, 0.99655,
        0.9999 , 0.9895 , 0.99829, 0.98681, 1.     , 1.     , 0.9945 ,
        0.9997 , 0.98044, 0.99697, 0.98279, 0.9997 , 0.99967]])

#### Phases

In [119]:
left_phases = np.array([
    d['left_defect_op_invariant'] for d in results
])

In [120]:
left_phases.shape

(20,)

In [121]:
np.round(left_phases, 3)

array([-0.576-0.28j , -0.43 -0.085j, -0.684-0.033j, -0.618-0.013j,
        0.021+0.092j, -0.84 -0.025j, -0.467+0.098j, -0.298-0.108j,
       -0.809+0.019j, -0.666-0.064j, -0.219-0.124j,  0.271+0.153j,
       -0.568+0.078j, -0.302-0.134j, -0.437+0.113j, -0.334+0.028j,
       -0.247+0.047j, -0.423+0.042j, -0.99 +0.002j, -0.454+0.024j])

In [122]:
right_phases = np.array([
    d['right_defect_op_invariant'] for d in results
])

In [123]:
right_phases.shape

(20,)

In [124]:
np.round(right_phases, 3)

array([-0.504-0.024j,  0.303+0.073j, -0.898+0.028j, -0.262-0.186j,
       -0.504-0.057j, -0.577+0.067j, -0.009+0.029j, -0.451-0.111j,
        0.191-0.222j, -0.553+0.068j, -0.254+0.02j , -0.332+0.145j,
       -0.305+0.106j, -0.457-0.182j,  0.029+0.048j, -0.217-0.002j,
        0.17 +0.101j,  0.078-0.121j, -0.062+0.171j,  0.124-0.12j ])

#### Cocyle information

In [125]:
np.array([
    d['cocycle_invariants'] for d in results
])

array([[-2.52774151e-01-3.22729154e-01j, -2.52699088e-01-2.45132128e-02j],
       [-1.77244639e-01-7.34720423e-02j, -7.28077870e-02-2.75815532e-02j],
       [-4.67264650e-01-4.58530257e-02j, -8.05946210e-01+5.00025197e-02j],
       [-3.81611330e-01-1.65076903e-02j, -2.03578286e-02-1.14166332e-01j],
       [ 7.79833455e-03-4.20325464e-03j, -2.50330179e-01-5.74846077e-02j],
       [-7.05253270e-01-4.18963156e-02j, -3.28577071e-01+7.74514934e-02j],
       [-2.08501371e-01+9.18217025e-02j,  6.97705458e-04+5.07293091e-04j],
       [-7.72816851e-02-6.41595867e-02j, -1.90721506e-01-9.99202341e-02j],
       [-6.52480532e-01+3.07046624e-02j,  1.32761198e-02+8.73238003e-02j],
       [-4.44554522e-01-8.90741269e-02j, -3.00585674e-01+7.49034835e-02j],
       [-3.20181625e-02-5.52701112e-02j, -6.54950215e-02+9.70959873e-03j],
       [-5.00533137e-02-8.31807751e-02j, -8.89251409e-02+9.64948889e-02j],
       [-3.16517414e-01+8.91450617e-02j, -8.21488635e-02+6.46356825e-02j],
       [-5.69482853e-02-8

In [126]:
cocycle_equation_array = np.array([d['cocycle_equation_output'] for d in results])
print(np.linalg.norm(cocycle_equation_array-1))
print(np.max(np.abs(cocycle_equation_array-1)))

93.32628770666271
29.66439330035673


Better!

In [127]:
bosonic_cocycle_equation_array = np.array([d['bosonic_cocycle_equation_output'] for d in results])

In [128]:
bosonic_cocycle_equation_array.shape

(20, 2, 2, 2, 2)

In [129]:
np.linalg.norm(bosonic_cocycle_equation_array[..., -1, -1, -1] + 1)

np.float64(5.5789222591942105)

In [130]:
bosonic_cocycle_equation_array_copy = bosonic_cocycle_equation_array.copy()
bosonic_cocycle_equation_array_copy[..., -1, -1, -1] = 1
np.linalg.norm(bosonic_cocycle_equation_array_copy - 1)

np.float64(5.26427302714587e-16)

In [131]:
np.round(
    np.array([d['cocycles'][:, 0, 0] for d in results])
, 3)

array([[-0.28 +0.576j,  0.024-0.503j],
       [ 0.086-0.43j ,  0.05 -0.274j],
       [-0.033+0.684j, -0.028-0.898j],
       [-0.013+0.618j,  0.219-0.261j],
       [-0.091+0.023j, -0.057+0.504j],
       [ 0.025-0.84j ,  0.067+0.577j],
       [-0.098-0.467j,  0.028+0.009j],
       [ 0.108-0.298j, -0.111+0.451j],
       [ 0.019+0.808j,  0.225+0.194j],
       [-0.066+0.67j , -0.068-0.552j],
       [-0.126+0.219j, -0.019-0.257j],
       [-0.153+0.271j,  0.145+0.332j],
       [-0.078-0.568j,  0.106+0.306j],
       [-0.144+0.279j,  0.183-0.457j],
       [-0.11 -0.437j,  0.048-0.026j],
       [-0.006-0.333j, -0.   +0.223j],
       [-0.047-0.247j,  0.102-0.168j],
       [ 0.042+0.423j,  0.133+0.069j],
       [ 0.002+0.99j , -0.174-0.07j ],
       [ 0.028+0.456j,  0.12 +0.125j]])

In [132]:
np.round(
    np.array([d['fermion_decoration_from_cocycle'] for d in results])
, 3)

array([[-0.621-0.027j, -0.253-0.098j],
       [-0.428-0.376j, -0.104-0.173j],
       [-0.73 -0.082j, -0.925-0.022j],
       [-0.577-0.036j, -0.324+0.084j],
       [ 0.101-0.169j, -0.253+0.123j],
       [-0.873-0.014j, -0.615-0.179j],
       [-0.621-0.099j,  0.053+0.252j],
       [ 0.137+0.091j, -0.43 -0.045j],
       [-0.85 +0.045j, -0.384-0.165j],
       [-0.461+0.002j, -0.616+0.186j],
       [-0.408+0.016j, -0.464-0.09j ],
       [-0.193-0.066j,  0.255+0.23j ],
       [ 0.125+0.02j , -0.319+0.045j],
       [-0.417-0.205j, -0.679+0.069j],
       [-0.07 +0.236j, -0.254-0.144j],
       [-0.533+0.041j, -0.145+0.122j],
       [-0.521-0.139j, -0.259+0.169j],
       [-0.591-0.264j, -0.058-0.408j],
       [-0.99 -0.j   , -0.077-0.055j],
       [-0.22 +0.218j, -0.356+0.193j]])

# Product state - 2 site projectors and defect operators

In [133]:
domains_dict = {
    'num_system_sites': 32,
    'left_projector_sites': list(range(4, 8)),
    'right_projector_sites': list(range(22, 28)),
    'num_projector_pad_sites': 2,
    'num_defect_sites': 4,
    'fdlu_depth': 2,
    'fdlu_offset': 0
}

In [134]:
product_psi = get_product_qu_tensor_network(domains_dict['num_system_sites'])

In [135]:
results = list()

for _ in tqdm(range(20)):
    current = find_invariants_via_projectors_from_random_state(
        product_psi,
        domains_dict,
        jw_even=True
    )
    results.append(current)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:16<00:00,  1.19it/s]


### Analyze results

#### Projector scores

In [136]:
np.round(np.array([
    d['left_proj_vec_results'][0][-1]
    for d in results
]), 3)

array([-0., -0.,  0., -0.,  0., -0., -0., -0., -0.,  0., -0.,  0., -0.,
        0., -0.,  0.,  0.,  0., -0.,  0.])

In [137]:
np.round(np.array([
    d['right_proj_vec_results'][0][-1]
    for d in results
]), 3)

array([-0.,  0.,  0., -0.,  0.,  0.,  0., -0.,  0.,  0., -0., -0., -0.,
        0.,  0., -0.,  0.,  0.,  0., -0.])

In [138]:
def get_schmidt_vals_ratio(schmidt_vals):
    if len(schmidt_vals.data) > 1:
        return schmidt_vals.data[1]/schmidt_vals.data[0]
    else:
        return 0

In [139]:
np.round(np.array([
    get_schmidt_vals_ratio(d['left_proj_vec_results'][1])
    for d in results
]), 3)

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [140]:
np.round(np.array([
    get_schmidt_vals_ratio(d['right_proj_vec_results'][1])
    for d in results
]), 3)

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

#### Purities

In [141]:
np.round(np.array(
    [d['cut_rho_purity'] for d in results]
), 3)

array([1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j,
       1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
       1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j])

In [142]:
np.round(np.array(
    [d['sub_cut_rho_trace'] for d in results]
), 3)

array([1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j,
       1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j,
       1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j])

In [143]:
np.round(np.array(
    [d['sub_cut_rho_purity'] for d in results]
), 3)

array([1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j,
       1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
       1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j])

Purities are the same, which one could likely prove.

In [144]:
np.round(np.array(
    [d['sub_cut_rho_purity'] for d in results]
), 3)

array([1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j,
       1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
       1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j])

#### Cut state results

In [145]:
np.array(
    [d['cut_state_fermion_parity'] for d in results]
)

array([1.+2.23972392e-18j, 1.-6.21544955e-19j, 1.+5.40992020e-19j,
       1.-1.37044483e-18j, 1.+5.80307016e-19j, 1.-8.31539996e-19j,
       1.-7.66025417e-19j, 1.+1.01278390e-18j, 1.-1.89415864e-18j,
       1.+1.62275939e-18j, 1.+1.02897655e-19j, 1.+1.24153504e-18j,
       1.+1.81165526e-18j, 1.-9.37661483e-21j, 1.+1.29110929e-18j,
       1.+3.76708745e-19j, 1.-1.07922676e-18j, 1.-8.37924048e-19j,
       1.+6.93381154e-19j, 1.-2.00976389e-18j])

So the FP of the cut state can be even or odd. Interesting!

In [146]:
np.round(np.array(
    [d['cut_overlap'] for d in results]
), 3)

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1.])

#### Defect op scores overlaps

In [147]:
np.round(np.array([
    [d['defect_ops_scores'][i][-1] for d in results]
    for i in range(3)
]), 5)

array([[1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1.],
       [1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1.],
       [1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1.]])

#### Phases

In [148]:
left_phases = np.array([
    d['left_defect_op_invariant'] for d in results
])

In [149]:
left_phases.shape

(20,)

In [150]:
np.round(left_phases, 3)

array([1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j,
       1.-0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
       1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j])

In [151]:
right_phases = np.array([
    d['right_defect_op_invariant'] for d in results
])

In [152]:
right_phases.shape

(20,)

In [153]:
np.round(right_phases, 3)

array([1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j,
       1.-0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j,
       1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j])

#### Cocyle information

In [154]:
np.array([
    d['cocycle_invariants'] for d in results
])

array([[1.-1.00941729e-16j, 1.+4.10741234e-17j],
       [1.+3.21705056e-16j, 1.+3.72232427e-16j],
       [1.-6.04092983e-18j, 1.-3.00055568e-17j],
       [1.+5.92791375e-16j, 1.+4.49708342e-16j],
       [1.+2.31759755e-16j, 1.+1.86739690e-16j],
       [1.+6.89458496e-17j, 1.-1.09866404e-16j],
       [1.-3.67805774e-16j, 1.-4.67832314e-16j],
       [1.+1.06827782e-16j, 1.+4.87849970e-16j],
       [1.+2.38187199e-16j, 1.+6.44068997e-17j],
       [1.+2.35491446e-16j, 1.+4.11454768e-17j],
       [1.-9.92609276e-17j, 1.+3.32654172e-17j],
       [1.-1.43684389e-15j, 1.-1.42083229e-15j],
       [1.+7.74126360e-17j, 1.+5.33830603e-17j],
       [1.+1.05345805e-16j, 1.-1.35897482e-16j],
       [1.+3.45372028e-16j, 1.+4.11662214e-16j],
       [1.+1.23792185e-15j, 1.+8.42341462e-16j],
       [1.+1.73694062e-16j, 1.+3.10975835e-16j],
       [1.+3.71907472e-17j, 1.+2.34581122e-16j],
       [1.-3.58766837e-16j, 1.-6.41136340e-17j],
       [1.-4.94470575e-16j, 1.-8.71119317e-16j]])

In [155]:
cocycle_equation_array = np.array([d['cocycle_equation_output'] for d in results])
np.linalg.norm(cocycle_equation_array-1)

np.float64(2.4564859455439353e-14)

In [156]:
bosonic_cocycle_equation_array = np.array([d['bosonic_cocycle_equation_output'] for d in results])

In [157]:
bosonic_cocycle_equation_array.shape

(20, 2, 2, 2, 2)

In [158]:
np.linalg.norm(bosonic_cocycle_equation_array - 1)

np.float64(3.187638529239956e-15)

In [159]:
np.round(
    np.array([d['cocycles'][:, 0, 0] for d in results])
, 3)

array([[ 1.+0.j,  1.-0.j],
       [-1.-0.j, -1.-0.j],
       [-1.-0.j, -1.+0.j],
       [-1.-0.j, -1.-0.j],
       [-1.-0.j, -1.+0.j],
       [ 1.+0.j,  1.-0.j],
       [ 1.-0.j,  1.-0.j],
       [ 1.+0.j,  1.+0.j],
       [-1.-0.j, -1.+0.j],
       [ 1.+0.j,  1.-0.j],
       [ 1.-0.j,  1.+0.j],
       [ 1.-0.j,  1.-0.j],
       [-1.-0.j, -1.-0.j],
       [-1.-0.j, -1.+0.j],
       [ 1.+0.j,  1.+0.j],
       [ 1.+0.j,  1.+0.j],
       [ 1.+0.j,  1.+0.j],
       [-1.-0.j, -1.-0.j],
       [ 1.-0.j,  1.-0.j],
       [ 1.-0.j,  1.-0.j]])

In [160]:
np.round(
    np.array([d['fermion_decoration_from_cocycle'] for d in results])
, 3)

array([[1.+0.j, 1.-0.j],
       [1.+0.j, 1.-0.j],
       [1.+0.j, 1.-0.j],
       [1.+0.j, 1.+0.j],
       [1.-0.j, 1.+0.j],
       [1.+0.j, 1.-0.j],
       [1.-0.j, 1.-0.j],
       [1.+0.j, 1.-0.j],
       [1.+0.j, 1.-0.j],
       [1.+0.j, 1.+0.j],
       [1.+0.j, 1.+0.j],
       [1.-0.j, 1.-0.j],
       [1.+0.j, 1.+0.j],
       [1.-0.j, 1.-0.j],
       [1.+0.j, 1.+0.j],
       [1.+0.j, 1.+0.j],
       [1.+0.j, 1.-0.j],
       [1.+0.j, 1.-0.j],
       [1.+0.j, 1.-0.j],
       [1.-0.j, 1.-0.j]])

# Conclusions
Looks good!